# Omnichannel Customer Analytics — RetailHive Dataset

**Auteur** : Nasserine Benchettara — Data Scientist | Customer Analytics  
**Dataset** : [RetailHive Pro — Kaggle](https://www.kaggle.com/datasets/ehsanzx/omni-channel-sales-dataset-20192025)  
**Periode** : 2019–2025  
**Derniere mise a jour** : 2025

---

## Contexte et objectifs

RetailHive est un retailer omnicanal operant sur 3 canaux : **Mobile**, **Online** et **Store**.  
Le dataset contient 12 000 transactions sur 6 ans.

**Probleme identifie** : le CA stagne depuis 2019 et la majorite des clients n'achete qu'une seule fois.

**3 objectifs analytiques :**

| Objectif | Question | Livrable |
|----------|----------|----------|
| 1. Diagnostiquer | Quel est l'etat reel de la base client ? | EDA + visualisations |
| 2. Segmenter | Quels groupes de clients identifier ? | 3 segments actionnables |
| 3. Predire | Quels one-shot peuvent devenir fideles ? | Liste de reactivation scoree |

---

## Structure du notebook

```
1. Setup & chargement des donnees
2. Exploration & diagnostic (EDA)
3. Feature engineering
4. Segmentation client
5. Modele ML — scoring de reactivation
6. Synthese & recommandations
```

---

## 1. Setup & chargement des donnees

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, roc_auc_score, RocCurveDisplay
import warnings
warnings.filterwarnings('ignore')

# ── Style global ──
plt.rcParams.update({
    'figure.facecolor' : 'white',
    'axes.facecolor'   : '#F9F9F9',
    'axes.grid'        : True,
    'grid.alpha'       : 0.35,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'font.family'      : 'sans-serif',
    'font.size'        : 11,
})

# ── Palette de couleurs ──
# Une couleur par segment — utilisee de facon coherente dans tout le notebook
PALETTE = {
    'Mobile'            : '#4ECBA4',
    'Online'            : '#7B6FF0',
    'Store'             : '#E8A44A',
    'Ambassadeur'       : '#4ECBA4',
    'Fidele Mono-canal' : '#7B6FF0',
    'One-shot'          : '#E06060',
    'US'                : '#5A9CF7',
    'Europe'            : '#E06060',
    'Asia'              : '#A8D86E',
}

In [ ]:
# ── Chargement ──
df = pd.read_csv('retailhive_10k.csv')
df['date'] = pd.to_datetime(df['date'])

# Conversion du rating texte en valeur numerique
rating_map = {'One':1, 'Two':2, 'Three':3, 'Four':4, 'Five':5}
df['rating_num'] = df['rating'].map(rating_map)

# Variables temporelles utiles pour l'analyse
df['year']    = df['date'].dt.year
df['month']   = df['date'].dt.month
df['quarter'] = df['date'].dt.to_period('Q').astype(str)

# Date de reference = lendemain du dernier achat dans le dataset
# Simule le 'aujourd hui' au moment de l'analyse
TODAY = df['date'].max() + pd.Timedelta(days=1)

print(f'Transactions  : {len(df):,}')
print(f'Clients       : {df["customer_id"].nunique():,}')
print(f'Periode       : {df["date"].min().date()} -> {df["date"].max().date()}')
print(f'Colonnes      : {df.columns.tolist()}')
df.head(3)

---
## 2. Exploration & Diagnostic (EDA)

Avant tout modelisation, on doit comprendre ce que contient le dataset.  
L'EDA repond a 3 questions : **la qualite des donnees est-elle acceptable ?**  
**Quels sont les signaux business visibles ?** **Quelles hypotheses peut-on formuler ?**

### 2.1 Qualite des donnees

In [ ]:
# ── Valeurs manquantes ──
nulls = df.isnull().sum()
print('Valeurs manquantes par colonne :')
print(nulls[nulls > 0] if nulls.sum() > 0 else '  Aucune valeur manquante')

# ── Doublons ──
dupes = df.duplicated(subset='transaction_id').sum()
print(f'\nDoublons sur transaction_id : {dupes}')

# ── Coherence revenue vs price * quantity ──
df['revenue_check'] = (df['price'] * df['quantity']).round(2)
ecart = (df['revenue'] - df['revenue_check']).abs()
print(f'\nCoherence revenue vs price*qty :')
print(f'  Ecart moyen : {ecart.mean():.4f} | Ecart max : {ecart.max():.4f}')
print('  => Leger ecart lie aux arrondis. Dataset coherent.')

# ── Resume statistique ──
print('\nStatistiques descriptives :')
df[['quantity','price','revenue','rating_num']].describe().round(2)

### 2.2 Evolution du CA par canal (2019–2025)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CA total par annee
ca_annuel = df.groupby('year')['revenue'].sum() / 1000
axes[0].bar(ca_annuel.index, ca_annuel.values,
            color='#4ECBA4', alpha=0.85, width=0.6, edgecolor='white')
axes[0].set_title('CA annuel (ke)', fontweight='bold')
axes[0].set_xlabel('Annee')
axes[0].set_ylabel('Revenue (ke)')
for i, v in enumerate(ca_annuel.values):
    axes[0].text(ca_annuel.index[i], v+0.3, f'{v:.0f}k',
                 ha='center', fontsize=10, fontweight='bold')

# CA par canal par annee
ca_canal = df.groupby(['year','channel'])['revenue'].sum().unstack() / 1000
for canal in ['Mobile','Online','Store']:
    axes[1].plot(ca_canal.index, ca_canal[canal], marker='o',
                 linewidth=2.5, markersize=6,
                 label=canal, color=PALETTE[canal])
axes[1].set_title('CA par canal (ke)', fontweight='bold')
axes[1].set_xlabel('Annee')
axes[1].legend()

plt.suptitle('Evolution du chiffre d affaires 2019-2025',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('viz_01_ca_annuel.png', dpi=150, bbox_inches='tight')
plt.show()

print('Insight : Le CA stagne autour de 180ke/an depuis 6 ans.')
print('         Pas de croissance organique visible.')
print('         C est le probleme #1 a resoudre.')

### 2.3 Comportement des clients — frequence d'achat

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution de la frequence
freq = df.groupby('customer_id')['transaction_id'].count()
freq_count = freq.value_counts().sort_index()

axes[0].bar(freq_count.index.astype(str), freq_count.values,
            color=['#4ECBA4','#7B6FF0','#E8A44A','#E06060'],
            alpha=0.85, edgecolor='white')
axes[0].set_title('Nombre de transactions par client', fontweight='bold')
axes[0].set_xlabel('Nb transactions')
axes[0].set_ylabel('Nb clients')
for i, (x, v) in enumerate(zip(freq_count.index, freq_count.values)):
    pct = v / freq.count() * 100
    axes[0].text(i, v+30, f'{pct:.1f}%',
                 ha='center', fontsize=11, fontweight='bold')

# Revenue par client
rev_client = df.groupby('customer_id')['revenue'].sum()
axes[1].hist(rev_client, bins=50, color='#7B6FF0',
             alpha=0.8, edgecolor='white')
axes[1].axvline(rev_client.median(), color='#E06060',
                linewidth=2, linestyle='--',
                label=f'Mediane : {rev_client.median():.0f}e')
axes[1].set_title('Distribution du revenue par client', fontweight='bold')
axes[1].set_xlabel('Revenue total (e)')
axes[1].legend()

plt.tight_layout()
plt.savefig('viz_02_frequence.png', dpi=150, bbox_inches='tight')
plt.show()

mono = freq_count[1]
multi = freq[freq >= 2].count()
ca_multi = df[df['customer_id'].isin(freq[freq>=2].index)]['revenue'].sum()
print(f'Clients avec 1 seul achat  : {mono:,} ({mono/freq.count()*100:.1f}%)')
print(f'Clients avec 2+ achats     : {multi:,} ({multi/freq.count()*100:.1f}%)')
print(f'CA genere par les 2+ achats: {ca_multi/df["revenue"].sum()*100:.1f}% du CA total')
print()
print('Insight majeur : 86% des clients n achetent qu une seule fois.')
print('La retention est l enjeu numero 1 de ce retailer.')

### 2.4 Satisfaction client (rating) par canal et region

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution des ratings par canal (barres empilees en %)
rating_canal = df.groupby(['channel','rating'])['transaction_id'].count().unstack()
rating_pct   = rating_canal.div(rating_canal.sum(axis=1), axis=0) * 100
rating_colors = ['#E06060','#E8A44A','#D3D1C7','#7B6FF0','#4ECBA4']
rating_pct.plot(kind='bar', stacked=True, ax=axes[0],
                color=rating_colors, alpha=0.9, edgecolor='white')
axes[0].set_title('Distribution des ratings par canal (%)', fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('% transactions')
axes[0].legend(title='Rating', labels=['1','2','3','4','5'],
               bbox_to_anchor=(1.05,1))
axes[0].tick_params(axis='x', rotation=0)

# Heatmap rating moyen region x canal
pivot = df.groupby(['region','channel'])['rating_num'].mean().unstack().round(2)
sns.heatmap(pivot, ax=axes[1], annot=True, fmt='.2f',
            cmap='RdYlGn', vmin=2.5, vmax=3.5,
            linewidths=0.5, cbar_kws={'label':'Rating moyen'})
axes[1].set_title('Rating moyen — Region x Canal', fontweight='bold')

plt.tight_layout()
plt.savefig('viz_03_rating.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Rating moyen global : {df["rating_num"].mean():.2f}/5')
print('Insight : Rating bas et uniforme (~2.9/5) sur tous les canaux et regions.')
print('         Probleme systemique, pas specifique a un canal.')

---
## 3. Feature Engineering

On transforme les 12 000 lignes de transactions en **1 ligne par client** avec des variables comportementales.  
C'est cette table qui servira de base pour la segmentation et le modele ML.

**Principe** : on construit 4 groupes de variables independants, puis on les fusionne.

### 3.1 Variables RFM

In [ ]:
# RFM : les 3 dimensions classiques de la valeur client
# R = Recency  : depuis combien de jours le client a achete (petit = actif)
# F = Frequency : combien de fois il a achete (grand = fidele)
# M = Monetary  : combien il a depense au total (grand = valeur elevee)

rfm = df.groupby('customer_id').agg(
    recency       = ('date',           lambda x: (TODAY - x.max()).days),
    frequency     = ('transaction_id', 'nunique'),
    monetary      = ('revenue',        'sum'),
    monetary_mean = ('revenue',        'mean'),
    first_purchase = ('date',          'min'),
).reset_index()

# Anciennete client en jours depuis le 1er achat
rfm['customer_age_days'] = (TODAY - rfm['first_purchase']).dt.days

print('Variables RFM construites :')
print(f'  Recency    | moy: {rfm["recency"].mean():.0f}j  | mediane: {rfm["recency"].median():.0f}j')
print(f'  Frequency  | moy: {rfm["frequency"].mean():.2f} | max: {rfm["frequency"].max()}')
print(f'  Monetary   | moy: {rfm["monetary"].mean():.0f}e | mediane: {rfm["monetary"].median():.0f}e')

### 3.2 Variables de comportement omnicanal

In [ ]:
# On veut caracteriser le comportement de chaque client sur les canaux :
# - Combien de canaux differents a-t-il utilises ?
# - Quel est son canal de preference ?
# - Quelle est sa repartition entre les 3 canaux ?

canal_features = df.groupby('customer_id').agg(
    nb_canaux     = ('channel', 'nunique'),
    canal_prefere = ('channel', lambda x: x.value_counts().idxmax()),
    pct_mobile    = ('channel', lambda x: (x == 'Mobile').mean()),
    pct_online    = ('channel', lambda x: (x == 'Online').mean()),
    pct_store     = ('channel', lambda x: (x == 'Store').mean()),
).reset_index()

print('Distribution du canal prefere :')
print(canal_features['canal_prefere'].value_counts())
print(f'\nClients utilisant 2+ canaux : {(canal_features["nb_canaux"]>=2).sum():,}')

### 3.3 Variables de satisfaction

In [ ]:
# On construit 3 variables a partir du rating :
# - rating_moyen   : satisfaction globale du client
# - rating_last    : son dernier ressenti (signal le plus recent)
# - rating_evolution : tendance (positif = s'ameliore, negatif = se degrade)

rating_features = df.sort_values('date').groupby('customer_id').agg(
    rating_moyen = ('rating_num', 'mean'),
    rating_first = ('rating_num', 'first'),
    rating_last  = ('rating_num', 'last'),
).reset_index()

rating_features['rating_evolution'] = (
    rating_features['rating_last'] - rating_features['rating_first']
)

print(f'Rating moyen global : {rating_features["rating_moyen"].mean():.2f}/5')
print(f'Clients avec evolution negative : {(rating_features["rating_evolution"] < 0).sum():,}')
print(f'Clients avec evolution positive : {(rating_features["rating_evolution"] > 0).sum():,}')

### 3.4 Fusion — table master_customer

In [ ]:
region_features = df.groupby('customer_id').agg(
    region = ('region', lambda x: x.value_counts().idxmax())
).reset_index()

# Fusion de toutes les variables en 1 ligne par client
master = (
    rfm
    .merge(canal_features,  on='customer_id')
    .merge(rating_features, on='customer_id')
    .merge(region_features, on='customer_id')
)

# Nettoyage des colonnes intermediaires
master = master.drop(columns=['first_purchase', 'rating_first'])

print(f'Table master_customer : {master.shape[0]:,} clients x {master.shape[1]} variables')
print(f'Valeurs manquantes    : {master.isnull().sum().sum()}')
print(f'\nColonnes : {master.columns.tolist()}')

### 3.5 Vue d'ensemble des variables construites

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

# 1 — Recency
axes[0].hist(master['recency'], bins=40, color='#4ECBA4', alpha=0.85, edgecolor='white')
axes[0].axvline(master['recency'].median(), color='#E06060', linewidth=2,
                linestyle='--', label=f'Mediane : {master["recency"].median():.0f}j')
axes[0].set_title('Recency (jours depuis dernier achat)', fontweight='bold')
axes[0].legend()

# 2 — Monetary
axes[1].hist(master['monetary'], bins=40, color='#7B6FF0', alpha=0.85, edgecolor='white')
axes[1].axvline(master['monetary'].median(), color='#E06060', linewidth=2,
                linestyle='--', label=f'Mediane : {master["monetary"].median():.0f}e')
axes[1].set_title('Monetary — Revenue total client (e)', fontweight='bold')
axes[1].legend()

# 3 — Canal prefere
cp = master['canal_prefere'].value_counts()
bars = axes[2].bar(cp.index, cp.values,
                   color=[PALETTE[c] for c in cp.index],
                   alpha=0.85, edgecolor='white')
axes[2].set_title('Canal prefere par client', fontweight='bold')
for bar, v in zip(bars, cp.values):
    axes[2].text(bar.get_x()+bar.get_width()/2, v+30,
                 f'{v:,}', ha='center', fontsize=10)

# 4 — Revenue mono vs omnicanal
rev_canal = master.groupby(master['nb_canaux']>=2)['monetary'].mean()
labels = ['Mono-canal', 'Multi-canal (2+)']
bars2 = axes[3].bar(labels, rev_canal.values,
                    color=['#D3D3D3','#4ECBA4'], alpha=0.85, edgecolor='white')
axes[3].set_title('Revenue moyen\nMono-canal vs Multi-canal', fontweight='bold')
for bar, v in zip(bars2, rev_canal.values):
    axes[3].text(bar.get_x()+bar.get_width()/2, v+1,
                 f'{v:.0f}e', ha='center', fontsize=12, fontweight='bold')

# 5 — Rating moyen
axes[4].hist(master['rating_moyen'], bins=20,
             color='#E8A44A', alpha=0.85, edgecolor='white')
axes[4].axvline(3, color='#E06060', linewidth=1.5,
                linestyle='--', label='Neutre (3/5)')
axes[4].set_title('Distribution du rating moyen', fontweight='bold')
axes[4].legend()

# 6 — Region
rg = master['region'].value_counts()
axes[5].bar(rg.index, rg.values,
            color=[PALETTE[r] for r in rg.index],
            alpha=0.85, edgecolor='white')
axes[5].set_title('Repartition geographique', fontweight='bold')
for i, v in enumerate(rg.values):
    axes[5].text(i, v+20, f'{v:,}', ha='center', fontsize=10)

plt.suptitle('Feature Engineering — Vue d ensemble des 16 variables client',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('viz_04_features.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Segmentation Client

### Pourquoi ne pas utiliser K-Means ici ?

K-Means est un algorithme de clustering qui cherche des groupes naturels dans les donnees.  
Il fonctionne bien quand les donnees ont des structures variees et distinctes.

**Ce qu'on observe sur ce dataset :**

In [ ]:
# Test K-Means pour observer ce qu'il produit sur ce dataset
features_km = [
    'recency', 'frequency', 'monetary',
    'nb_canaux', 'pct_mobile', 'pct_online', 'pct_store',
    'rating_moyen', 'customer_age_days'
]

X = master[features_km].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Silhouette score pour k=2 a k=6
print('Silhouette scores par k :')
print('(rappel : plus proche de 1 = groupes mieux separes)')
print()
scores = {}
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    s = silhouette_score(X_scaled, labels)
    scores[k] = s
    print(f'  k={k} | silhouette={s:.4f}')

best_k = max(scores, key=scores.get)
print(f'\nMeilleur k mathematiquement : k={best_k} (silhouette={scores[best_k]:.4f})')

In [ ]:
# Regardons ce que k=4 (meilleur silhouette) produit concretement
km4 = KMeans(n_clusters=4, random_state=42, n_init=10)
master['cluster_k4'] = km4.fit_predict(X_scaled)

profil_k4 = master.groupby('cluster_k4').agg(
    nb_clients  = ('customer_id', 'count'),
    recency_moy = ('recency',     'mean'),
    freq_moy    = ('frequency',   'mean'),
    revenue_moy = ('monetary',    'mean'),
    nb_canaux   = ('nb_canaux',   'mean'),
    pct_mobile  = ('pct_mobile',  'mean'),
    pct_online  = ('pct_online',  'mean'),
    pct_store   = ('pct_store',   'mean'),
).round(2)

print('Profil des 4 clusters K-Means :')
print(profil_k4.to_string())
print()
print('OBSERVATION :')
print('Les clusters 1, 2 et 3 ont des profils quasi identiques :')
print('  recency ~1 255j, frequency ~1.04, monetary ~106e')
print('  La seule difference : la repartition entre canaux (mobile / online / store)')
print()
print('CONCLUSION :')
print('K-Means ne trouve pas 4 comportements differents.')
print('Il separe les one-shot par canal utilise, pas par comportement.')
print('Ce resultat n est pas actionnable business.')
print('=> On abandonne K-Means et on utilise une segmentation metier explicite.')

# Nettoyage
master = master.drop(columns=['cluster_k4'])

### 4.1 Segmentation metier — 2 axes independants

On construit la segmentation sur 2 axes conceptuellement distincts :

**Axe 1 — Fidelite** : est-ce que le client revient ?  
Mesure par `frequency`. Seuil : `frequency >= 2`.

**Axe 2 — Comportement omnicanal** : utilise-t-il plusieurs canaux ?  
Mesure par `nb_canaux`. Seuil : `nb_canaux >= 2`.

> Ces deux axes sont **independants** : un client fidele n'est pas necessairement omnicanal,  
> et un client omnicanal n'est pas necessairement fidele.

In [ ]:
# Construction des deux variables binaires
master['est_fidele']    = (master['frequency'] >= 2).astype(int)
master['est_omnicanal'] = (master['nb_canaux'] >= 2).astype(int)

# Crosstab : combien de clients dans chaque combinaison ?
ct = pd.crosstab(
    master['est_fidele'],
    master['est_omnicanal'],
    rownames=['Fidele (freq>=2)'],
    colnames=['Omnicanal (nb_canaux>=2)']
)
print('Structure des donnees — croisement fidelite x omnicanal :')
print(ct)
print()
print('CONSTAT STRUCTUREL :')
print('La case [0,1] contient 0 clients.')
print('Avec 1 seule transaction, on ne peut utiliser qu un seul canal.')
print('Omnicanal implique necessairement plusieurs achats dans ce dataset.')
print('=> 3 segments reels, pas 4.')

### 4.2 Attribution des 3 segments

In [ ]:
def attribuer_segment(row):
    """
    Regles de segmentation metier explicites.

    Ambassadeur       : fidele ET utilise plusieurs canaux
    Fidele Mono-canal : fidele mais reste sur un seul canal
    One-shot          : 1 seul achat (85% de la base)
    """
    if row['est_fidele'] == 1 and row['est_omnicanal'] == 1:
        return 'Ambassadeur'
    elif row['est_fidele'] == 1 and row['est_omnicanal'] == 0:
        return 'Fidele Mono-canal'
    else:
        return 'One-shot'

master['segment'] = master.apply(attribuer_segment, axis=1)

print('Repartition des 3 segments :')
print()
for seg, n in master['segment'].value_counts().items():
    pct = n / len(master) * 100
    barre = '#' * int(pct / 2)
    print(f'  {seg:<22} {n:>5} clients  {pct:.1f}%  {barre}')

### 4.3 Profil detaille des segments

In [ ]:
profil = master.groupby('segment').agg(
    nb_clients    = ('customer_id',   'count'),
    recency_moy   = ('recency',       'mean'),
    frequency_moy = ('frequency',     'mean'),
    monetary_moy  = ('monetary',      'mean'),
    panier_moy    = ('monetary_mean', 'mean'),
    nb_canaux_moy = ('nb_canaux',     'mean'),
    rating_moy    = ('rating_moyen',  'mean'),
    ca_total      = ('monetary',      'sum'),
).round(2)

profil['pct_base'] = (profil['nb_clients'] / len(master) * 100).round(1)
profil['pct_ca']   = (profil['ca_total'] / master['monetary'].sum() * 100).round(1)

print('=' * 75)
print('PROFIL DES 3 SEGMENTS')
print('=' * 75)
print(profil[['nb_clients','pct_base','recency_moy','frequency_moy',
              'monetary_moy','panier_moy','nb_canaux_moy',
              'rating_moy','ca_total','pct_ca']].to_string())
print('=' * 75)
print()
amb = profil.loc['Ambassadeur','monetary_moy']
fid = profil.loc['Fidele Mono-canal','monetary_moy']
print(f'Ambassadeur vs Fidele Mono-canal :')
print(f'  Revenue moy Ambassadeur   : {amb:.0f}e')
print(f'  Revenue moy Fidele Mono   : {fid:.0f}e')
print(f'  Ecart                     : {abs(amb-fid):.0f}e')
print()
print('=> La FIDELITE cree la valeur, pas l omnicanal seul.')
print('   Un client mono-canal fidele vaut autant qu un ambassadeur.')

### 4.4 Visualisation des segments

In [ ]:
segments_order = ['Ambassadeur', 'Fidele Mono-canal', 'One-shot']
colors_list    = [PALETTE[s] for s in segments_order]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

# 1 — Nombre de clients
nb = [master[master['segment']==s].shape[0] for s in segments_order]
bars = axes[0].bar(segments_order, nb, color=colors_list, alpha=0.85, width=0.5)
axes[0].set_title('Nombre de clients par segment', fontweight='bold')
for bar, v in zip(bars, nb):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+50,
                 f'{v:,}\n({v/len(master)*100:.1f}%)',
                 ha='center', fontsize=10, fontweight='bold')
axes[0].tick_params(axis='x', rotation=10)

# 2 — Revenue moyen
rev = [master[master['segment']==s]['monetary'].mean() for s in segments_order]
bars2 = axes[1].bar(segments_order, rev, color=colors_list, alpha=0.85, width=0.5)
axes[1].set_title('Revenue moyen par client (e)', fontweight='bold')
for bar, v in zip(bars2, rev):
    axes[1].text(bar.get_x()+bar.get_width()/2, v+1,
                 f'{v:.0f}e', ha='center', fontsize=11, fontweight='bold')
diff = abs(rev[0]-rev[1])
axes[1].annotate(f'Ecart : {diff:.0f}e\nFidelite > Omnicanal',
                 xy=(0.5,(rev[0]+rev[1])/2), xytext=(0.55, rev[0]*0.55),
                 ha='center', fontsize=9, color='#555',
                 arrowprops=dict(arrowstyle='->', color='#aaa'))
axes[1].tick_params(axis='x', rotation=10)

# 3 — CA total
ca = [master[master['segment']==s]['monetary'].sum()/1000 for s in segments_order]
bars3 = axes[2].bar(segments_order, ca, color=colors_list, alpha=0.85, width=0.5)
axes[2].set_title('CA total genere (ke)', fontweight='bold')
for bar, v in zip(bars3, ca):
    axes[2].text(bar.get_x()+bar.get_width()/2, v+1,
                 f'{v:.0f}ke\n({v/sum(ca)*100:.1f}%)',
                 ha='center', fontsize=10, fontweight='bold')
axes[2].tick_params(axis='x', rotation=10)

# 4 — Apport de chaque dimension
categories = ['One-shot\n(ni fidele\nni omnicanal)',
              'Fidele\nMono-canal',
              'Ambassadeur\n(fidele +\nomnicanal)']
values = [master[master['segment']==s]['monetary'].mean()
          for s in segments_order[::-1]]
bars4 = axes[3].bar(categories, values,
                    color=[PALETTE['One-shot'],
                           PALETTE['Fidele Mono-canal'],
                           PALETTE['Ambassadeur']],
                    alpha=0.85, width=0.5)
axes[3].set_title('Apport reel de chaque dimension\nFidelite vs Omnicanal', fontweight='bold')
for bar, v in zip(bars4, values):
    axes[3].text(bar.get_x()+bar.get_width()/2, v+1,
                 f'{v:.0f}e', ha='center', fontsize=11, fontweight='bold')
u1 = (values[1]-values[0])/values[0]*100
u2 = (values[2]-values[1])/values[1]*100
axes[3].annotate(f'+{u1:.0f}%\nfidelite',
                 xy=(1,values[1]), xytext=(0.45, values[1]*0.8),
                 ha='center', fontsize=10, color=PALETTE['Fidele Mono-canal'],
                 fontweight='bold',
                 arrowprops=dict(arrowstyle='->', color=PALETTE['Fidele Mono-canal']))
axes[3].annotate(f'+{u2:.0f}%\nomnicanal',
                 xy=(2,values[2]), xytext=(1.55, values[2]*0.8),
                 ha='center', fontsize=10, color=PALETTE['Ambassadeur'],
                 fontweight='bold',
                 arrowprops=dict(arrowstyle='->', color=PALETTE['Ambassadeur']))
axes[3].tick_params(axis='x', rotation=0)

plt.suptitle('Segmentation Client — 3 segments actionnables',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('viz_05_segments.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.5 Plan d'action par segment

In [ ]:
print('=' * 70)
print('PLAN D ACTION PAR SEGMENT')
print('=' * 70)

actions = {
    'Ambassadeur': {
        'nb'      : master[master['segment']=='Ambassadeur'].shape[0],
        'ca_pct'  : 17.7,
        'enjeu'   : 'Maintenir et recompenser leur engagement',
        'action'  : 'Programme VIP, offres exclusives, avant-premieres',
        'canal'   : 'Email personnalise + push mobile + experience magasin',
        'priorite': 'HAUTE',
    },
    'Fidele Mono-canal': {
        'nb'      : master[master['segment']=='Fidele Mono-canal'].shape[0],
        'ca_pct'  : 8.2,
        'enjeu'   : 'Maintenir la fidelite, encourager exploration d un 2e canal',
        'action'  : 'Offre exclusive sur un nouveau canal pour inciter a tester',
        'canal'   : 'Sur leur canal prefere actuel',
        'priorite': 'HAUTE',
    },
    'One-shot': {
        'nb'      : master[master['segment']=='One-shot'].shape[0],
        'ca_pct'  : 74.0,
        'enjeu'   : 'Declencher le 2e achat — seuil critique vers la fidelite',
        'action'  : 'Email J+14 post-achat, recommandation produit complementaire',
        'canal'   : 'Canal de leur 1er achat (ne pas forcer la migration)',
        'priorite': 'MOYENNE — fort volume, 5% conversion = impact majeur',
    },
}

for seg, info in actions.items():
    print(f'\n-- {seg.upper()} ({info["nb"]:,} clients — {info["ca_pct"]}% du CA) --')
    print(f'   Enjeu    : {info["enjeu"]}')
    print(f'   Action   : {info["action"]}')
    print(f'   Canal    : {info["canal"]}')
    print(f'   Priorite : {info["priorite"]}')
print('\n' + '=' * 70)

---
## 5. Modele ML — Scoring de reactivation

### Objectif

Parmi les 8 873 clients one-shot, identifier ceux qui ont le plus de chances
de faire un 2e achat pour concentrer les efforts de reactivation.

### Regle anti-data leakage

On construit les features **uniquement a partir du 1er achat**.  
Utiliser des donnees ulterieures (frequency, nb_canaux...) pour predire
si le client revient = **data leakage** = AUC artificiel de 1.0, modele inutile en production.

### 5.1 Construction du dataset ML

In [ ]:
# Clients qui ont fait au moins 2 achats = la cible positive
freq_total = df.groupby('customer_id')['transaction_id'].count()
multi_clients = freq_total[freq_total >= 2].index

# Premier achat de chaque client = seules donnees disponibles
# au moment ou on doit decider si on envoie une campagne de reactivation
first_purchase = df.sort_values('date').groupby('customer_id').first().reset_index()

model_df = first_purchase[['customer_id','revenue','rating_num',
                            'channel','region']].copy()
model_df.columns = ['customer_id','revenue_first','rating_first',
                    'channel_first','region_first']

# Variable cible : est-ce que ce client est revenu ?
model_df['est_revenu'] = model_df['customer_id'].isin(multi_clients).astype(int)

# Encodage des variables categorielles en numerique
model_df['canal_enc']  = model_df['channel_first'].map({'Mobile':0,'Online':1,'Store':2})
model_df['region_enc'] = model_df['region_first'].map({'US':0,'Europe':1,'Asia':2})

print('Distribution de la variable cible :')
for val, n in model_df['est_revenu'].value_counts().items():
    label = 'A fait un 2e achat' if val==1 else 'Pas de 2e achat'
    print(f'  {label:<20} : {n:>5} ({n/len(model_df)*100:.1f}%)')
print()
print('Dataset desequilibre : 14% positifs — comportement normal pour du churn/retention.')

### 5.2 Entrainement et evaluation

In [ ]:
FEATURES = ['revenue_first', 'rating_first', 'canal_enc', 'region_enc']
TARGET   = 'est_revenu'

X = model_df[FEATURES]
y = model_df[TARGET]

# Split stratifie : preserve la proportion 14%/86% dans train et test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = GradientBoostingClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

auc    = roc_auc_score(y_test, y_proba)
cv_auc = cross_val_score(model, X, y, cv=5, scoring='roc_auc')

print(f'AUC-ROC      : {auc:.3f}')
print(f'AUC CV 5fold : {cv_auc.mean():.3f} +/- {cv_auc.std():.3f}')
print()
print(classification_report(y_test, y_pred))

### 5.3 Interpretation du resultat

**AUC ~ 0.50 : le modele ne predit pas mieux qu'un tirage aleatoire.**

Ce n'est pas un echec — c'est une information importante.

**Pourquoi ?**  
Le 1er achat contient seulement 4 informations : revenue, rating, canal, region.  
Ces 4 variables n'expliquent pas pourquoi un client revient.

Le retour depend de facteurs **absents du dataset** :  
- Qualite de l'email de suivi post-achat  
- Experience de livraison et SAV  
- Offre de reactivation recue  
- Comportement de navigation sur le site

**Recommandation au client** : enrichir la collecte de donnees comportementales  
(ouverture email, pages visitees, interactions SAV) pour ameliorer la prediction.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Courbe ROC
RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[0],
                                  color='#7B6FF0', name='Gradient Boosting')
axes[0].plot([0,1],[0,1], linestyle='--', color='gray', label='Aleatoire (AUC=0.50)')
axes[0].set_title(f'Courbe ROC — AUC={auc:.3f}', fontweight='bold')
axes[0].legend()

# Feature importance
fi = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
fi_labels = {'revenue_first':'Revenue 1er achat',
             'rating_first' :'Rating 1er achat',
             'canal_enc'    :'Canal utilise',
             'region_enc'   :'Region client'}
fi.index = [fi_labels[i] for i in fi.index]
fi.plot(kind='barh', ax=axes[1], color='#4ECBA4', alpha=0.85)
axes[1].set_title('Importance des variables', fontweight='bold')
axes[1].set_xlabel('Importance relative')

plt.tight_layout()
plt.savefig('viz_06_modele.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.4 Scoring des one-shot — liste de reactivation

In [ ]:
# Meme avec AUC faible, les probabilites predites permettent de PRIORISER.
# Contacter les clients avec la proba la plus elevee = meilleur ROI
# qu'une campagne envoyee aleatoirement a tous les one-shot.

one_shot_data = model_df[model_df['est_revenu'] == 0].copy()
one_shot_data['proba_retour'] = model.predict_proba(
    one_shot_data[FEATURES]
)[:, 1]

# Segmentation en 3 niveaux de priorite par tercile
one_shot_data['priorite'] = pd.qcut(
    one_shot_data['proba_retour'], q=3,
    labels=['Faible priorite','Priorite moyenne','Haute priorite']
)

prio_order  = ['Haute priorite','Priorite moyenne','Faible priorite']
prio_colors = {'Haute priorite':'#4ECBA4',
               'Priorite moyenne':'#E8A44A',
               'Faible priorite':'#E06060'}

print('Repartition par priorite :')
print(one_shot_data.groupby('priorite').agg(
    nb_clients   = ('customer_id',   'count'),
    revenue_moy  = ('revenue_first', 'mean'),
    rating_moy   = ('rating_first',  'mean'),
    canal_top    = ('channel_first', lambda x: x.value_counts().idxmax()),
    proba_moy    = ('proba_retour',  'mean'),
).round(2).to_string())

### 5.5 Estimation de l'impact business

In [ ]:
print('=' * 65)
print('ESTIMATION DE L IMPACT BUSINESS')
print('=' * 65)

val_fidele   = master[master['est_fidele']==1]['monetary'].mean()
val_one_shot = master[master['segment']=='One-shot']['monetary'].mean()
gain         = val_fidele - val_one_shot

hypotheses = {'Haute priorite':0.05, 'Priorite moyenne':0.02, 'Faible priorite':0.005}

print(f'Valeur client fidele (moy)  : {val_fidele:.0f}e')
print(f'Valeur client one-shot      : {val_one_shot:.0f}e')
print(f'Gain par conversion         : {gain:.0f}e')
print()
print(f'{"Priorite":<22} {"Clients":>8} {"Taux conv":>10} {"Conv":>8} {"CA recuperable":>16}')
print('-' * 68)
total = 0
for p in prio_order:
    n    = one_shot_data[one_shot_data['priorite']==p].shape[0]
    taux = hypotheses[p]
    conv = int(n * taux)
    ca   = conv * gain
    total += ca
    print(f'{p:<22} {n:>8,} {taux*100:>9.1f}% {conv:>8,} {ca:>15,.0f}e')
print('-' * 68)
print(f'{"TOTAL":<22} {"":>8} {"":>10} {"":>8} {total:>15,.0f}e')

### 5.6 Export de la liste de reactivation

In [ ]:
liste = one_shot_data[
    ['customer_id','channel_first','region_first',
     'revenue_first','rating_first','proba_retour','priorite']
].sort_values('proba_retour', ascending=False)

liste.to_csv('liste_reactivation_oneshot.csv', index=False)
master.to_csv('master_customer_segmented.csv', index=False)

print('Fichiers exportes :')
print(f'  master_customer_segmented.csv  : {len(master):,} clients x {master.shape[1]} variables')
print(f'  liste_reactivation_oneshot.csv : {len(liste):,} one-shot scores et priorises')
print()
print('Top 10 clients a contacter en priorite :')
print(liste.head(10).to_string(index=False))

---
## 6. Synthese & Recommandations

### Ce qu'on a appris

In [ ]:
print('=' * 70)
print('SYNTHESE DE L ANALYSE OMNICANALE — RETAILHIVE')
print('=' * 70)

print()
print('DIAGNOSTIC (EDA)')
print('  - CA stable autour de 180ke/an depuis 2019 : pas de croissance')
print(f'  - {(master["frequency"]==1).sum()/len(master)*100:.1f}% des clients n achetent qu une seule fois')
print('  - Rating moyen de 2.9/5 uniforme sur tous les canaux et regions')
print('  - Probleme systemique, pas specifique a un canal')

print()
print('SEGMENTATION')
for seg in ['Ambassadeur','Fidele Mono-canal','One-shot']:
    sub = master[master['segment']==seg]
    pct_ca = sub['monetary'].sum()/master['monetary'].sum()*100
    print(f'  {seg:<22} : {len(sub):>5} clients ({len(sub)/len(master)*100:.1f}%) | '
          f'rev moy {sub["monetary"].mean():.0f}e | {pct_ca:.1f}% du CA')

print()
print('LECON CLE : La fidelite cree la valeur, pas l omnicanal seul.')
print('  Fidele Mono-canal (214e) = Ambassadeur (218e) en revenue moyen.')
print('  Priorite #1 : convertir les one-shot en clients fideles.')

print()
print('MODELE ML')
print('  - AUC ~ 0.50 : le 1er achat ne suffit pas a predire le retour')
print('  - Ce n est pas un probleme d algorithme, c est un probleme de donnees')
print('  - Recommandation : enrichir la collecte (email, navigation, SAV)')
print('  - Meme imparfait, le scoring permet de prioriser les campagnes')

print()
print('ACTIONS PRIORITAIRES')
print('  1. Proteger les Ambassadeurs et Fideles (14% base, 26% du CA)')
print('  2. Email reactivation J+14 sur les one-shot haute priorite')
print('  3. Enrichir la collecte de donnees pour ameliorer la prediction')
print('  4. Investiguer la cause du rating bas (2.9/5) — SAV ? Livraison ?')
print('=' * 70)

---

## Fichiers produits

| Fichier | Description |
|---------|-------------|
| `master_customer_segmented.csv` | Table client enrichie — 1 ligne par client, 18 variables + segment |
| `liste_reactivation_oneshot.csv` | 8 873 one-shot scores et priorises pour campagne de reactivation |
| `viz_01_ca_annuel.png` | Evolution du CA par annee et canal |
| `viz_02_frequence.png` | Distribution de la frequence et du revenue |
| `viz_03_rating.png` | Satisfaction par canal et region |
| `viz_04_features.png` | Vue d ensemble des 16 variables construites |
| `viz_05_segments.png` | Visualisation des 3 segments |
| `viz_06_modele.png` | Courbe ROC et importance des variables |

---

## Stack technique

```
Python 3.10+
pandas, numpy
matplotlib, seaborn
scikit-learn (StandardScaler, KMeans, GradientBoostingClassifier)
```

---

*Nasserine Benchettara — Data Scientist | Customer Analytics*  
*[LinkedIn](https://linkedin.com) | [Malt](https://malt.fr)*